## **Practice 8: Model Analysis and Evaluation**

In [ ]:
# ============================================================================================================
# TASK 8: LLM-AS-A-JUDGE EVALUATION CON MÉTRICAS AUTOMÁTICAS
# Evaluación automatizada usando múltiples modelos de Bedrock como jueces + métricas automáticas
# ============================================================================================================

import json
import time
import pandas as pd
from pathlib import Path
from datetime import datetime
from typing import List, Dict
import numpy as np
from tqdm import tqdm

from langchain_aws import ChatBedrock
from langchain_core.prompts import PromptTemplate
from evaluate import load

# ============================================================================================================
# CONFIGURACIÓN
# ============================================================================================================
REGION = "us-east-1"
RESULTS_DIR = Path("./evaluation_results")
RESULTS_DIR.mkdir(exist_ok=True)

# Modelos de Bedrock a evaluar
# Modelos de Bedrock a evaluar
MODELS_TO_EVALUATE = [
    "arn:aws:bedrock:us-east-1:867344470723:inference-profile/us.anthropic.claude-3-5-haiku-20241022-v1:0",
    "arn:aws:bedrock:us-east-1:867344470723:inference-profile/global.amazon.nova-2-lite-v1:0",
    "arn:aws:bedrock:us-east-1:867344470723:inference-profile/us.anthropic.claude-3-haiku-20240307-v1:0",
]

# Modelo juez
JUDGE_MODEL = "arn:aws:bedrock:us-east-1:867344470723:inference-profile/us.anthropic.claude-3-5-haiku-20241022-v1:0"

# Costos por 1000 tokens (ajusta según pricing real)
MODEL_COSTS = {
    "arn:aws:bedrock:us-east-1:867344470723:inference-profile/us.anthropic.claude-3-5-haiku-20241022-v1:0": {"input": 0.00025, "output": 0.00125},
    "arn:aws:bedrock:us-east-1:867344470723:inference-profile/us.anthropic.claude-3-haiku-20240307-v1:0": {"input": 0.00025, "output": 0.00125},
    "arn:aws:bedrock:us-east-1:867344470723:inference-profile/global.amazon.nova-2-lite-v1:0": {"input": 0.00006, "output": 0.00024},
}
EVAL_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# Cargar métricas automáticas
print("📊 Cargando métricas automáticas...")
bleu_metric = load("bleu")
rouge_metric = load("rouge")
bertscore_metric = load("bertscore")
print("✅ Métricas cargadas\n")

# ============================================================================================================
# PROMPT DEL JUEZ
# ============================================================================================================

JUDGE_PROMPT_TEMPLATE = """Eres un evaluador experto de respuestas de modelos de lenguaje.

Tu tarea es evaluar qué tan bien la respuesta del sistema responde a la pregunta del usuario.

Pregunta: {question}
Respuesta esperada: {ground_truth}
Respuesta del sistema: {answer}

Evalúa la respuesta en una escala de 1 a 4:
1: Terrible - completamente irrelevante o muy parcial
2: Pobre - pierde algunos aspectos clave de la pregunta
3: Buena - proporciona apoyo pero podría mejorar
4: Excelente - relevante, directa, detallada

Proporciona tu evaluación en el siguiente formato:

Evaluación: (tu razonamiento para la calificación)
Calificación: (tu calificación, un número entre 1 y 4)

Evaluación:"""

judge_prompt = PromptTemplate(
    input_variables=["question", "ground_truth", "answer"],
    template=JUDGE_PROMPT_TEMPLATE
)

# ============================================================================================================
# BENCHMARK DATASET (50 preguntas - mismo que antes)
# ============================================================================================================

def create_benchmark():
    """Crea benchmark de 50 preguntas"""
    return [
        # Matemáticas (10)
        {"domain": "math", "question": "¿Cuánto es 15 + 27?", "answer": "42"},
        {"domain": "math", "question": "¿Cuál es la raíz cuadrada de 144?", "answer": "12"},
        {"domain": "math", "question": "¿Cuánto es 8 x 7?", "answer": "56"},
        {"domain": "math", "question": "¿Cuánto es 100 - 37?", "answer": "63"},
        {"domain": "math", "question": "¿Cuánto es 50 / 5?", "answer": "10"},
        {"domain": "math", "question": "¿Cuánto es 2 elevado a 5?", "answer": "32"},
        {"domain": "math", "question": "¿Cuánto es 15% de 200?", "answer": "30"},
        {"domain": "math", "question": "¿Cuál es el perímetro de un cuadrado de lado 5?", "answer": "20"},
        {"domain": "math", "question": "¿Cuántos grados tiene un triángulo?", "answer": "180 grados"},
        {"domain": "math", "question": "¿Cuánto es 3/4 más 1/4?", "answer": "1"},
        
        # Ciencia (10)
        {"domain": "science", "question": "¿Cuál es el símbolo químico del agua?", "answer": "H2O"},
        {"domain": "science", "question": "¿Cuántos planetas hay en el sistema solar?", "answer": "8 planetas"},
        {"domain": "science", "question": "¿Qué gas respiramos?", "answer": "Oxígeno"},
        {"domain": "science", "question": "¿Cuál es el planeta más grande?", "answer": "Júpiter"},
        {"domain": "science", "question": "¿A qué velocidad viaja la luz?", "answer": "300,000 kilómetros por segundo"},
        {"domain": "science", "question": "¿Qué es la fotosíntesis?", "answer": "Proceso donde las plantas producen alimento usando luz solar"},
        {"domain": "science", "question": "¿Cuántos huesos tiene el cuerpo humano adulto?", "answer": "206 huesos"},
        {"domain": "science", "question": "¿Qué es el ADN?", "answer": "Ácido desoxirribonucleico, material genético"},
        {"domain": "science", "question": "¿Cuál es la temperatura de ebullición del agua?", "answer": "100 grados Celsius"},
        {"domain": "science", "question": "¿Qué fuerza mantiene los planetas en órbita?", "answer": "La gravedad"},
        
        # Historia (10)
        {"domain": "history", "question": "¿En qué año llegó Colón a América?", "answer": "1492"},
        {"domain": "history", "question": "¿Quién fue el primer presidente de Estados Unidos?", "answer": "George Washington"},
        {"domain": "history", "question": "¿En qué año cayó el Muro de Berlín?", "answer": "1989"},
        {"domain": "history", "question": "¿Quién pintó la Mona Lisa?", "answer": "Leonardo da Vinci"},
        {"domain": "history", "question": "¿En qué año terminó la Segunda Guerra Mundial?", "answer": "1945"},
        {"domain": "history", "question": "¿Quién fue Napoleón Bonaparte?", "answer": "Emperador francés y líder militar"},
        {"domain": "history", "question": "¿En qué siglo ocurrió la Revolución Francesa?", "answer": "Siglo XVIII (1789)"},
        {"domain": "history", "question": "¿Qué fue la Guerra Fría?", "answer": "Conflicto político e ideológico entre USA y URSS"},
        {"domain": "history", "question": "¿Quién descubrió la penicilina?", "answer": "Alexander Fleming"},
        {"domain": "history", "question": "¿En qué año se fundó la ONU?", "answer": "1945"},
        
        # Geografía (10)
        {"domain": "geography", "question": "¿Cuál es la capital de Francia?", "answer": "París"},
        {"domain": "geography", "question": "¿Cuál es el río más largo del mundo?", "answer": "El río Amazonas"},
        {"domain": "geography", "question": "¿En qué continente está Egipto?", "answer": "África"},
        {"domain": "geography", "question": "¿Cuál es el océano más grande?", "answer": "El Océano Pacífico"},
        {"domain": "geography", "question": "¿Cuál es la montaña más alta del mundo?", "answer": "El Monte Everest"},
        {"domain": "geography", "question": "¿Cuántos continentes hay?", "answer": "7 continentes"},
        {"domain": "geography", "question": "¿Cuál es el país más grande del mundo?", "answer": "Rusia"},
        {"domain": "geography", "question": "¿Dónde está la Torre Eiffel?", "answer": "París, Francia"},
        {"domain": "geography", "question": "¿Cuál es el desierto más grande?", "answer": "El desierto del Sahara"},
        {"domain": "geography", "question": "¿En qué país están las pirámides de Giza?", "answer": "Egipto"},
        
        # Tecnología (10)
        {"domain": "tech", "question": "¿Qué significa CPU?", "answer": "Central Processing Unit"},
        {"domain": "tech", "question": "¿Quién fundó Microsoft?", "answer": "Bill Gates y Paul Allen"},
        {"domain": "tech", "question": "¿Qué es Python?", "answer": "Un lenguaje de programación"},
        {"domain": "tech", "question": "¿Qué significa HTML?", "answer": "HyperText Markup Language"},
        {"domain": "tech", "question": "¿Qué es una GPU?", "answer": "Graphics Processing Unit"},
        {"domain": "tech", "question": "¿Qué es AWS?", "answer": "Amazon Web Services"},
        {"domain": "tech", "question": "¿Qué es Machine Learning?", "answer": "Aprendizaje automático"},
        {"domain": "tech", "question": "¿Qué significa API?", "answer": "Application Programming Interface"},
        {"domain": "tech", "question": "¿Qué es Git?", "answer": "Sistema de control de versiones"},
        {"domain": "tech", "question": "¿Qué es una base de datos?", "answer": "Sistema para almacenar datos"},
    ]

# ============================================================================================================
# INFERENCIA CON MODELOS
# ============================================================================================================

def estimate_tokens(text: str) -> int:
    """Estimación aproximada de tokens (1 token ≈ 4 caracteres)"""
    return len(text) // 4

def run_inference_for_model(model_id: str, questions: List[str]) -> List[Dict]:
    """Ejecuta inferencia con un modelo específico"""
    print(f"\n🤖 Ejecutando inferencia con {model_id.split('.')[-1][:30]}...")
    
    llm = ChatBedrock(model_id=model_id, region_name=REGION)
    results = []
    
    for question in tqdm(questions, desc="Inference"):
        start = time.time()
        
        try:
            response = llm.invoke([{"role": "user", "content": f"Responde de forma breve y precisa: {question}"}])
            response_text = response.content
            
            # Estimar tokens
            input_tokens = estimate_tokens(question)
            output_tokens = estimate_tokens(response_text)
            
        except Exception as e:
            response_text = f"Error: {str(e)}"
            input_tokens = 0
            output_tokens = 0
        
        latency = time.time() - start
        
        results.append({
            "model": model_id,
            "question": question,
            "response": response_text,
            "latency": latency,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens
        })
    
    return results

# ============================================================================================================
# EVALUACIÓN CON LLM JUDGE
# ============================================================================================================

def judge_response(question: str, ground_truth: str, answer: str) -> Dict:
    """Usa LLM como juez para evaluar una respuesta"""
    judge_llm = ChatBedrock(model_id=JUDGE_MODEL, region_name=REGION)
    
    prompt_text = judge_prompt.format(
        question=question,
        ground_truth=ground_truth,
        answer=answer
    )
    
    try:
        result = judge_llm.invoke([{"role": "user", "content": prompt_text}])
        evaluation = result.content
        
        # Extraer calificación
        import re
        if "Calificación:" in evaluation:
            score_text = evaluation.split("Calificación:")[1].strip()
            numbers = re.findall(r'\d+', score_text)
            score = int(numbers[0]) if numbers else 0
        else:
            score = 0
        
        return {"evaluation": evaluation, "score": score}
    except Exception as e:
        return {"evaluation": f"Error: {str(e)}", "score": 0}

def evaluate_all_responses(benchmark: List[Dict], all_results: List[Dict]) -> pd.DataFrame:
    """Evalúa todas las respuestas usando el juez"""
    print("\n⚖️  Evaluando respuestas con LLM Judge...")
    
    evaluation_data = []
    
    for result in tqdm(all_results, desc="Judging"):
        question_idx = next(i for i, q in enumerate(benchmark) if q["question"] == result["question"])
        ground_truth = benchmark[question_idx]["answer"]
        domain = benchmark[question_idx]["domain"]
        
        judgment = judge_response(result["question"], ground_truth, result["response"])
        
        evaluation_data.append({
            "eval_id": EVAL_TIMESTAMP,
            "model": result["model"],
            "domain": domain,
            "question": result["question"],
            "ground_truth": ground_truth,
            "response": result["response"],
            "latency": result["latency"],
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "judge_evaluation": judgment["evaluation"],
            "judge_score": judgment["score"]
        })
    
    return pd.DataFrame(evaluation_data)

# ============================================================================================================
# MÉTRICAS AUTOMÁTICAS
# ============================================================================================================

def calculate_automatic_metrics(predictions: List[str], references: List[str]) -> Dict:
    """Calcula métricas automáticas (BLEU, ROUGE, BERTScore)"""
    print("📊 Calculando métricas automáticas...")
    
    # BLEU
    bleu_results = bleu_metric.compute(
        predictions=predictions,
        references=[[r] for r in references]
    )
    
    # ROUGE
    rouge_results = rouge_metric.compute(
        predictions=predictions,
        references=references
    )
    
    # BERTScore - FORZAR A CPU PARA EVITAR ERROR DE CUDA
    bertscore_results = bertscore_metric.compute(
        predictions=predictions,
        references=references,
        lang="es",
        model_type="distilbert-base-multilingual-cased",
        device="cpu"  # ✅ FORZAR CPU
    )
    
    return {
        "bleu": bleu_results["bleu"],
        "rouge1": rouge_results["rouge1"],
        "rouge2": rouge_results["rouge2"],
        "rougeL": rouge_results["rougeL"],
        "bertscore_precision": np.mean(bertscore_results["precision"]),
        "bertscore_recall": np.mean(bertscore_results["recall"]),
        "bertscore_f1": np.mean(bertscore_results["f1"]),
    }

# ============================================================================================================
# ANÁLISIS DE COSTOS
# ============================================================================================================

def calculate_costs(eval_df: pd.DataFrame) -> pd.DataFrame:
    """Calcula costos por modelo"""
    print("\n💰 Calculando costos...")
    
    cost_data = []
    
    for model in eval_df["model"].unique():
        model_data = eval_df[eval_df["model"] == model]
        
        if model in MODEL_COSTS:
            total_input_tokens = model_data["input_tokens"].sum()
            total_output_tokens = model_data["output_tokens"].sum()
            
            input_cost = (total_input_tokens / 1000) * MODEL_COSTS[model]["input"]
            output_cost = (total_output_tokens / 1000) * MODEL_COSTS[model]["output"]
            total_cost = input_cost + output_cost
            
            cost_data.append({
                "eval_id": EVAL_TIMESTAMP,
                "model": model,
                "total_input_tokens": total_input_tokens,
                "total_output_tokens": total_output_tokens,
                "input_cost_usd": round(input_cost, 6),
                "output_cost_usd": round(output_cost, 6),
                "total_cost_usd": round(total_cost, 6),
                "cost_per_query_usd": round(total_cost / len(model_data), 6)
            })
    
    return pd.DataFrame(cost_data)

# ============================================================================================================
# GUARDAR RESULTADOS
# ============================================================================================================

def save_results(eval_df: pd.DataFrame, benchmark: List[Dict]):
    """Guarda resultados en CSVs"""
    print("\n💾 Guardando resultados...")
    
    # 1. Evaluaciones completas
    eval_file = RESULTS_DIR / f"evaluations_{EVAL_TIMESTAMP}.csv"
    eval_df.to_csv(eval_file, index=False)
    print(f"✅ Evaluaciones: {eval_file}")
    
    # 2. Métricas LLM Judge por modelo
    judge_metrics = eval_df.groupby("model").agg({
        "judge_score": ["mean", "std", "min", "max"],
        "latency": ["mean", "std"]
    }).round(3)
    judge_metrics.columns = ["judge_avg", "judge_std", "judge_min", "judge_max", "latency_avg", "latency_std"]
    judge_metrics = judge_metrics.reset_index()
    judge_metrics["eval_id"] = EVAL_TIMESTAMP
    
    # 3. Métricas automáticas por modelo
    auto_metrics_data = []
    for model in eval_df["model"].unique():
        model_data = eval_df[eval_df["model"] == model]
        predictions = model_data["response"].tolist()
        references = model_data["ground_truth"].tolist()
        
        auto_metrics = calculate_automatic_metrics(predictions, references)
        auto_metrics["model"] = model
        auto_metrics["eval_id"] = EVAL_TIMESTAMP
        auto_metrics_data.append(auto_metrics)
    
    auto_metrics_df = pd.DataFrame(auto_metrics_data)
    
    # Combinar métricas
    combined_metrics = judge_metrics.merge(auto_metrics_df, on=["model", "eval_id"])
    metrics_file = RESULTS_DIR / f"model_metrics_{EVAL_TIMESTAMP}.csv"
    combined_metrics.to_csv(metrics_file, index=False)
    print(f"✅ Métricas combinadas: {metrics_file}")
    
    # 4. Métricas por dominio
    domain_metrics = eval_df.groupby(["model", "domain"]).agg({
        "judge_score": "mean",
        "latency": "mean"
    }).round(3).reset_index()
    domain_metrics.columns = ["model", "domain", "avg_judge_score", "avg_latency"]
    domain_metrics["eval_id"] = EVAL_TIMESTAMP
    
    domain_file = RESULTS_DIR / f"domain_metrics_{EVAL_TIMESTAMP}.csv"
    domain_metrics.to_csv(domain_file, index=False)
    print(f"✅ Métricas por dominio: {domain_file}")
    
    # 5. Análisis de costos
    cost_df = calculate_costs(eval_df)
    cost_file = RESULTS_DIR / f"costs_{EVAL_TIMESTAMP}.csv"
    cost_df.to_csv(cost_file, index=False)
    print(f"✅ Análisis de costos: {cost_file}")
    
    return eval_file, metrics_file, domain_file, cost_file

# ============================================================================================================
# MAIN
# ============================================================================================================

def main():
    print("="*100)
    print("⚖️  LLM-AS-A-JUDGE EVALUATION CON MÉTRICAS AUTOMÁTICAS")
    print("="*100)
    print(f"📊 Modelos a evaluar: {len(MODELS_TO_EVALUATE)}")
    print(f"⚖️  Modelo juez: {JUDGE_MODEL}")
    print(f"📈 Métricas: LLM Judge, BLEU, ROUGE, BERTScore")
    print(f"💰 Análisis de costos incluido")
    
    # Crear benchmark
    benchmark = create_benchmark()
    questions = [q["question"] for q in benchmark]
    print(f"\n📝 Preguntas en benchmark: {len(questions)}")
    
    # Ejecutar inferencia para todos los modelos
    all_results = []
    for model_id in MODELS_TO_EVALUATE:
        results = run_inference_for_model(model_id, questions)
        all_results.extend(results)
    
    # Evaluar con juez
    eval_df = evaluate_all_responses(benchmark, all_results)
    
    # Guardar resultados
    files = save_results(eval_df, benchmark)
    
    # Mostrar resumen
    print("\n" + "="*100)
    print("📊 RESUMEN DE EVALUACIÓN")
    print("="*100)
    summary = eval_df.groupby("model")["judge_score"].agg(["mean", "std", "min", "max"]).round(2)
    print(summary)
    
    print("\n💰 RESUMEN DE COSTOS")
    print("="*100)
    cost_summary = pd.read_csv(files[3])
    print(cost_summary[["model", "total_cost_usd", "cost_per_query_usd"]].to_string(index=False))
    
    print("\n" + "="*100)
    print("✅ EVALUACIÓN COMPLETADA")
    print("="*100)
    print("📁 Archivos CSV generados:")
    for f in files:
        print(f"   - {f}")
    print("\n💡 Ejecuta: python task-8-ui.py")
    print("   Para visualizar los resultados en Gradio")
    print("="*100)

if __name__ == "__main__":
    main()

📊 Cargando métricas automáticas...
✅ Métricas cargadas

⚖️  LLM-AS-A-JUDGE EVALUATION CON MÉTRICAS AUTOMÁTICAS
📊 Modelos a evaluar: 3
⚖️  Modelo juez: arn:aws:bedrock:us-east-1:867344470723:inference-profile/us.anthropic.claude-3-5-haiku-20241022-v1:0
📈 Métricas: LLM Judge, BLEU, ROUGE, BERTScore
💰 Análisis de costos incluido

📝 Preguntas en benchmark: 50

🤖 Ejecutando inferencia con claude-3-5-haiku-20241022-v1:0...


Inference: 100%|██████████| 50/50 [00:00<00:00, 49624.99it/s]



🤖 Ejecutando inferencia con nova-2-lite-v1:0...


Inference: 100%|██████████| 50/50 [00:00<00:00, 61464.01it/s]



🤖 Ejecutando inferencia con claude-3-haiku-20240307-v1:0...


...


⚖️  Evaluando respuestas con LLM Judge...


Judging: 100%|██████████| 150/150 [00:00<00:00, 278.90it/s]



💾 Guardando resultados...
✅ Evaluaciones: evaluation_results/evaluations_20260209_215653.csv
📊 Calculando métricas automáticas...


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1211.11it/s, Materializing param=transformer.layer.5.sa_layer_norm.weight]   
DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RuntimeError: CUDA error: CUBLAS_STATUS_INVALID_VALUE when calling `cublasSgemmStridedBatched( handle, opa, opb, m, n, k, &alpha, a, lda, stridea, b, ldb, strideb, &beta, c, ldc, stridec, num_batches)`